In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
orders_df=spark.sql("select * from dev.raw_uki_ecommerce.ecommerce_orders")
product_df=spark.sql("select * from dev.raw_uki_ecommerce.ecommerce_products")

# Create stg dataframes ready to ingest

In [0]:
# this block explodes and flattens all orders data into a single row per order
exploded_df=orders_df.withColumn("customer_exploded",F.explode(F.array("customer"))) \
    .withColumn("items_exploded",F.explode("items")) \
        .withColumn("payment_exploded",F.explode(F.array("payment")))\
            .withColumn("shipping_exploded",F.explode(F.array("shipping"))) \
                .withColumn("gift_exploded",F.explode(F.array("gift"))) \
                    .withColumn("totals_exploded",F.explode(F.array("totals"))) \
                        .withColumn("metadata_exploded",F.explode(F.array("metadata"))) 

display(exploded_df)

# Create stg tables 


In [0]:
# create stg order table 
orderstg=exploded_df.select(
    F.col("order_id"),
    F.col("order_date"),
    F.col("order_status"),
    F.col("channel"),
    F.col("gift_exploded.is_gift"),
    F.col("gift_exploded.gift_message"),
    F.col("totals_exploded.subtotal"),
    F.col("totals_exploded.tax"),
    F.col("totals_exploded.shipping_cost"),
    F.col("totals_exploded.promo_discount"),
    F.col("totals_exploded.grand_total"),
    F.col("customer_exploded.customer_id"),
    F.col("items_exploded.product_id"),
    F.col("payment_exploded.transaction_id"),
    F.col("shipping_exploded.tracking_id"),
    F.col("metadata_exploded.session_id")
)
display(orderstg)


In [0]:
customers_sort=exploded_df.select(
    F.col("customer_exploded.*")

)

customers_stg=customers_sort.withColumn("address_exploded",F.explode(F.array(F.col("address")))) \
    .select(
        F.col("customer_id"),
        F.col("name"),
        F.col("email"),
        F.col("address_exploded.*"),
        F.col("loyalty_tier")
    )
display(customers_stg)

In [0]:
shippingstg=exploded_df.select(
    F.col("shipping_exploded.*")
)

display(shippingstg)